# 🎙️ Bahgat zero-shot grid: VoiceTut-TTS vs base OmniVoice vs IndexTTS 2.5

**Goal:** find out — in one ~30–60 min Colab session, before any training — whether
a zero-shot Egyptian-Arabic engine already sounds like Bahgat Saber well enough
to use directly (or as the donor voice for RVC). **Run this notebook FIRST,
before `bahgat_rvc_gpt-sovits_training.ipynb`.**

**Order of tests (do them in this order):** (1) **VoiceTut-TTS first** — an
OmniVoice fine-tune on ~380 h of Egyptian podcast speech, `arz`, Apache-2.0
(`mohammedaly22/VoiceTut-TTS`, PyPI `voicetut-tts` 0.1.1). (2) **Base
OmniVoice second** (`k2-fsa/OmniVoice`) — the control that shows whether
VoiceTut's Egyptian fine-tuning is what makes the difference. (3) **IndexTTS
2.5 third** — Arabic (AR) + emotion/duration control, the best prosody surface.
(4) **`voicetut-egyptian-emotion` donor (#9)** — a VoiceTut fine-tune with
`[shouting]` / `[furious, …]` tags, **donor only** (single voice, cloning
unsupported — it can never BE the Bahgat voice; RVC re-timbres it afterwards).

**The 9-cell grid** (audit §4 — same **5 fixed test sentences** (§3) in every
cell, only one knob changes per cell; `guidance_scale` is held **constant at
2.0** for all VoiceTut/OmniVoice cells, see §4 helpers):

| # | Model | language | ref clip | ref_text | speed | num_step | What it answers |
|---|---|---|---|---|---|---|---|
| 1 | VoiceTut | `arz` | fast rant | manual | 1.0 | 32 | Does Egyptian fine-tuning fix dialect + pace? |
| 2 | VoiceTut | `arz` | laughing clip | manual | 1.0 | 32 | Does ref style transfer to output? (+ `[laughter]` tag test) |
| 3 | VoiceTut | `arb` | same as #1 | manual | 1.0 | 32 | Is `arz` vs `arb` the dialect difference? |
| 4 | Base OmniVoice | `arz` | same as #1 | manual | 1.0 | 32 | Is VoiceTut actually better than base? |
| 5 | VoiceTut | `arz` | same as #1 | **auto (omit)** | 1.0 | 32 | How much does Whisper mis-transcription cost? |
| 6 | VoiceTut | `arz` | same as #1 | manual | **1.25** | 32 | Does speed help, or only trim? |
| 7 | VoiceTut | `arz` | same as #1 | manual | 1.0 | **16** | Quality cost of fast sampling? |
| 8 | IndexTTS 2.5 | `AR` | rant (timbre) + shouting clip (emotion ref) | — | `duration_factor` 0.8 | — | Best emotion/duration control? |
| 9 | `voicetut-egyptian-emotion` | `arz` | **none** (tag-only, single voice) | — | — | — | Best shouting donor for RVC? (**donor only**) |

**Decision rule (fill the §6 scorecard by ear, then apply this):**
- If **#1 / #2 score close on timbre AND pace** (≥4/5 on both) → **skip RVC
  training entirely**, or use VoiceTut as the **RVC donor** (it replaces calm
  Edge TTS Shakir as the prosody source, RVC only re-timbres).
- If VoiceTut clearly beats base (#4) on dialect → the `arz` fine-tune is the
  difference; keep `arz` everywhere.
- If #5 (auto ref_text) ≈ #1 → manual transcripts are not critical. If #5 is
  clearly worse → always use hand-verified `ref_text`.
- If #6 (speed 1.25) paces better without hurting dialect → use speed 1.25.
  If #7 (step 16) ≈ #1 → use 16 for faster batch generation.
- If #8 wins on emotion/pace → IndexTTS 2.5 is the donor / direct voice.
- If **#9 wins on shouting prosody** → use it as the **RVC donor** for
  rant/shouting lines (RVC re-timbre to Bahgat; never use #9's own voice as
  Bahgat — it is a single synthetic voice, cloning unsupported).
- If **none** is close → run the RVC training notebook (the Drive upload in §2
  is already done — the curated zip is the same file).

**Prerequisites:** `bahgat_dataset_curated.zip` (182 MB, 361 clips, 38.11 min)
uploaded to the **root of Google Drive** ("My Drive"). Runtime → **T4 GPU**
(verify §0 shows a T4). Keep the tab open. Everything here is free and local
to your Drive + the Colab VM — nothing is uploaded anywhere else.

Sources: audit `research_notes/bahgat-voice-pro-techniques-2026-20261126-0625/report.md`
§4; VoiceTut PyPI `voicetut-tts` (install + `VoiceTutTTS` API) and HF
`mohammedaly22/VoiceTut-TTS-GGUF` (always `--language arz`, 5–10 s refs);
OmniVoice PyPI `lahgtna-omnivoice`; IndexTTS 2.5 repo `index-tts/index-tts`
(`duration_factor` 0.5–2.0, `emo_audio_prompt`, `lang="AR"`); emotion donor
HF `masterofaudio2077/voicetut-egyptian-emotion` (card + Kaggle tutorial,
usage verified against the live model card 2026-11-26); final pre-Colab
verification `research_notes/bahgat-pre-colab-final-check-2026-20261126-0738/report.md`
§2.2/§3a (frozen ref_texts, fixed 5-sentence set, `guidance_scale` 2.0,
`[laughter]` logging, IndexTTS pitfalls, donor cell #9).


## §0 — GPU check

Must show a T4 (15 GB VRAM). VoiceTut needs ~2.5–3.9 GB, base OmniVoice ~6 GB,
IndexTTS 2.5 ~6–8 GB — a T4 is comfortable for all three. Do not run this on a
CPU-only runtime.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('STOP: no GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, then re-run.')


## §1 — Setup: VoiceTut-TTS + base OmniVoice

**VoiceTut-TTS (PyTorch, the Colab-native path — use this):**
```bash
pip install git+https://github.com/k2-fsa/OmniVoice.git   # OmniVoice backbone (not on PyPI)
pip install voicetut-tts                                   # PyPI 0.1.1
```
```python
from voicetut_tts import VoiceTutTTS
tts = VoiceTutTTS.from_pretrained('mohammedaly22/VoiceTut-TTS')
tts.synthesize(text, ref_audio='ref.wav', ref_text='...', output='out.wav',
               num_step=32, speed=1.0)
```
VoiceTut is an Egyptian-only fine-tune, so `arz` is its native behaviour (its
own card: "Always pass `--language arz`"). The `synthesize()` call below passes
`language='arz'` only if the installed version exposes that parameter — if it
does not, the model still speaks Egyptian by default and only test #3 (the
`arb` comparison) is skipped with a printed warning.

**Base OmniVoice** (the control): `pip install omnivoice`,
`OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0',
dtype=torch.float16)` then `model.generate(text, ref_audio, ref_text,
language_id='arz', num_step=32)` — same API as the earlier comparison notebook.

**IndexTTS 2.5 is installed separately in §4.8** (its `uv sync` takes ~10 min)
so a heavy install can never block tests #1–#7.

**Alternative (not used here):** the GGUF package
`mohammedaly22/VoiceTut-TTS-GGUF` (`voicetut-tts-q8_0.gguf`, 1.26 GB) runs
through `audio.cpp` (`audiocpp_cli --task tts --family omnivoice --language
arz ...`) — same weights, no Python stack, but building audio.cpp on Colab is
slower than `pip install voicetut-tts`. Only try it if the PyTorch install
fails.

Do **not** `pip install torch` — Colab already ships torch + CUDA.


In [ ]:
# --- Install VoiceTut-TTS + base OmniVoice + helpers ---------------------------
# OmniVoice backbone first (VoiceTut depends on it), then the voicetut-tts package.
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git 2>&1 | tail -n 3
!pip install -q voicetut-tts soundfile 2>&1 | tail -n 3
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1; ffmpeg -version | head -n 1

import importlib.metadata as _md
for _pkg in ('voicetut-tts', 'omnivoice', 'soundfile'):
    try:
        print(_pkg, _md.version(_pkg))
    except Exception as e:
        print(_pkg, 'NOT INSTALLED:', e)


## §2 — Reference clips (from the curated set)

**⚠️ READ THIS BEFORE USING ANY TRANSCRIPT AS `ref_text` — it matters more
than any knob below.**

The transcripts under each clip are **hand-corrected from garbled Whisper
output** (Whisper on fast, shouting, profanity-heavy Egyptian dialect is
heavily garbled — see `training/curated/REFERENCE_CANDIDATES.md`). They are
best-effort Egyptian orthography, **NOT verified word-for-word**. A wrong
`ref_text` teaches the model the **wrong phonemes** for that reference clip —
the model will then pronounce other words wrong too.

**So: listen to each clip in Colab and fix its transcript BY EAR before it is
used as `ref_text`.** The cell below prints the transcripts and saves them to
`/content/refs/ref_transcripts.json` — edit that file (or the `REF_*_TEXT`
variables in the §3 cell) if you hear different words. If in doubt about a
word, **omit `ref_text`** for that test and let the model auto-transcribe
(Whisper normalises toward MSA — worse dialect, but no wrong-phoneme teaching).

**Clip lengths vs the grid's "5 s":** the audit's grid says ~5 s refs, but the
three candidates are **7.68–9.30 s** — still inside OmniVoice/VoiceTut's
documented **3–10 s** window (VoiceTut GGUF card: clean 5–10 s reference).
They are used **whole on purpose**: trimming to exactly 5 s would cut the
clips mid-sentence and **invalidate the transcripts above**. If you still want
exactly 5 s, the trim command is in the cell below (commented out) — but then
you **must** re-transcribe the trimmed 5 s by ear before using it as
`ref_text`.

| Role | Clip | Dur. | Mean F0 | RMS | Hand-corrected transcript (**VERIFY BY EAR**) |
|---|---|---|---|---|---|
| 🔥 Fast loud rant → tests #1, #3, #4, #5, #6, #7 + IndexTTS #8 (timbre) | `bahgat_0621.wav` (train-accident live 8/11/2026) | 9.30 s | 193.2 Hz | −19.6 dB | "تمام، وإن طائفتان من المؤمنين اقتتلوا... أولًا الطائفة... الجيش المصري بقيادة السيسي والإخوان. قف عندك! قف عندك!" |
| 😂 Laughing / joke → test #2 | `bahgat_0470.wav` (funny-komanda compilation) | 7.68 s | 172.9 Hz | −15.2 dB | "إنت عارف إسرائيل في 73 كانت على بعد كام كيلو من القاهرة؟ من ميدان التحرير على بعد كام كيلو!" |
| 📞 Live-call style → call-style line ref + spare | `bahgat_0265.wav` (lagan call-ins) | 8.30 s | 154.3 Hz | −11.8 dB | "سلّم البلد يا سيسي، هو إحنا أول شعب في التاريخ الإنساني يقوم بثورة ضد محتل داخلي!" |

Alternates (same folder, if a clip turns out unusable): rant `bahgat_0615.wav`
(9.28 s, F0 212.2 Hz, medium confidence); laughing `bahgat_0545.wav` (7.70 s,
F0 182.0 Hz); call `bahgat_0283.wav` (8.52 s, F0 164.2 Hz).

**FROZEN 2026-11-26 (final-check §3a):** the three transcripts below are
**frozen constants** in the next cell, copied verbatim from
`training/curated/REFERENCE_CANDIDATES.md` (hand-corrected Egyptian
orthography, exact words including profanity/laughter as transcribed). Do not
re-type or "improve" them — if you hear a different word by ear, correct the
constant in the cell AND note the correction in the scorecard, so the frozen
set stays reproducible.


In [ ]:
# --- Fetch the 3 reference clips from the curated zip on Drive ---------------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json, os, zipfile
import soundfile as sf

ZIP_CANDS = ['/content/drive/MyDrive/bahgat_dataset_curated.zip',
             '/content/drive/MyDrive/bahgat_dataset.zip']  # old name, fallback only
ZIP = next((z for z in ZIP_CANDS if os.path.exists(z)), None)
assert ZIP, 'bahgat_dataset_curated.zip NOT FOUND in Drive root — upload it to My Drive first (see docs/colab-start-checklist.md).'
print('zip:', ZIP, f'({round(os.path.getsize(ZIP)/1e6, 1)} MB)')

os.makedirs('/content/refs', exist_ok=True)
WANTED = {'bahgat_0621.wav': 'rant', 'bahgat_0470.wav': 'laughing', 'bahgat_0265.wav': 'call'}
with zipfile.ZipFile(ZIP) as z:
    names = z.namelist()
    for wav, role in WANTED.items():
        hit = next((n for n in names if n.endswith('/' + wav) or n == wav), None)
        assert hit, f'{wav} not in zip!'
        z.extract(hit, '/content/refs_tmp')
        os.replace(f'/content/refs_tmp/{hit}', f'/content/refs/{wav}')
        print(f'{role:8s} {wav} <- {hit}')

# Optional: trim to exactly 5 s (grid spec) — OFF by default, see §2 warning:
# trimming invalidates the transcripts below (they cover the FULL clip).
# !ffmpeg -y -i /content/refs/bahgat_0621.wav -t 5 /content/refs/bahgat_0621_5s.wav

REF_RANT  = '/content/refs/bahgat_0621.wav'
REF_LAUGH = '/content/refs/bahgat_0470.wav'
REF_CALL  = '/content/refs/bahgat_0265.wav'

# FROZEN ref_texts (2026-11-26, final-check §3a) — copied VERBATIM from
# training/curated/REFERENCE_CANDIDATES.md. Do NOT re-type or paraphrase:
# these exact strings are the frozen constants for the whole grid.
# VERIFY BY EAR before use (see §2): listen to each clip in Colab and fix
# a word here ONLY if you actually hear a different word — then note the
# correction in the §6 scorecard. If in doubt about a word, omit ref_text
# for that test (auto-transcribe) instead of guessing.
REF_RANT_TEXT  = "تمام، وإن طائفتان من المؤمنين اقتتلوا... أولًا الطائفة... الجيش المصري بقيادة السيسي والإخوان. قف عندك! قف عندك!"
REF_LAUGH_TEXT = "إنت عارف إسرائيل في 73 كانت على بعد كام كيلو من القاهرة؟ من ميدان التحرير على بعد كام كيلو!"
REF_CALL_TEXT  = "سلّم البلد يا سيسي، هو إحنا أول شعب في التاريخ الإنساني يقوم بثورة ضد محتل داخلي!"

for label, path, text in [('RANT ', REF_RANT, REF_RANT_TEXT),
                          ('LAUGH', REF_LAUGH, REF_LAUGH_TEXT),
                          ('CALL ', REF_CALL, REF_CALL_TEXT)]:
    info = sf.info(path)
    print(f'{label} {os.path.basename(path)}: {info.duration:.2f} s, {info.samplerate} Hz, {info.channels} ch')
    print(f'      ref_text: {text}')

json.dump({'bahgat_0621.wav': REF_RANT_TEXT, 'bahgat_0470.wav': REF_LAUGH_TEXT,
           'bahgat_0265.wav': REF_CALL_TEXT},
          open('/content/refs/ref_transcripts.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)
print('saved /content/refs/ref_transcripts.json — edit it + the variables above if the transcripts are wrong by ear.')

# Frozen-set guard: the constants above must be non-empty (a silent empty
# ref_text would make the model auto-transcribe without telling you).
assert REF_RANT_TEXT.strip() and REF_LAUGH_TEXT.strip() and REF_CALL_TEXT.strip(), 'a frozen ref_text is empty — restore it from REFERENCE_CANDIDATES.md.'
print('frozen ref_texts OK (3/3 non-empty, verbatim from REFERENCE_CANDIDATES.md).')


## §3 — Test sentences (FIVE fixed, same in every cell)

Five SHORT Egyptian lines in Bahgat's fast/loud register — **the same five in
every grid cell (#1–#9), no cell invents its own wording**. Each is exactly
**10 words** (counted in the cell below, so §5 can compute words/s without
Whisper). One per style the bot actually needs: debate, joke/laughing,
shouting rant, fast profanity-heavy, code-switching (Arabic/English).

| Key | Style | Sentence (10 words) | What it tests |
|---|---|---|---|
| `debate` | 🗣️ Debate | "الدليل من القرآن والسنة واضح، واللي يعترض يجيب دليله قدامنا" | MSA-leaning religious debate register — does the model stay Egyptian? |
| `joke` | 😂 Joke / laughing | "قالك مرة واحد راح يشتري عقل، قالوله خلص من زمان!" | Joke pacing + `[laughter]` tag (test #2 only) |
| `rant` | 🔥 Shouting rant | "فوقوا بقى، البلد دي مش لعبة في إيديكم والناس زهقت!" | Loud/fast rant prosody — the bot's core style |
| `profanity` | 🤬 Fast profanity-heavy | "إنت وسخ يا معفن، فاكر نفسك مين يا ابن الكلب؟" | Profanity-heavy fast speech — mispronunciations show up here first |
| `codeswitch` | 🌐 Code-switching | "يا جماعة الـ live ده نار والـ chat بيضحك عليكم!" | Arabic/English mixing — does the model keep the English words intelligible? |

**`[laughter]` tag — open test, NOT an assumed feature:** only test #2
prefixes the joke line with `"[laughter] "`. Plain VoiceTut/OmniVoice
documentation does **not** promise bracketed tags produce laughter — the
emotion-model card warns OmniVoice's own `generate` "counts the English tag
as speech and often reads it aloud" (final-check §2.2). So **log the tag
behaviour per cell in the §6 scorecard: read aloud / ignored / laughed**.
If the tag comes out spoken as a word ("لافتر") instead of a laugh, that tells
you tags do not survive for Egyptian in that engine — do not assume it works.

**Raw donor WAVs:** every output below is saved raw to `/content/grid/` and
exported by §7 **before any RVC pass** — the grid outputs ARE the donor
material. Nothing here is re-timbred inside this notebook.


In [ ]:
# --- Test sentences: FIVE fixed, 10 words each (word count drives the §5 words/s metric) ---
# FROZEN 2026-11-26 (final-check §3a): same five sentences in EVERY grid cell
# (#1-#9). Do not reword per cell — comparability depends on identical input.
TEST_SENTENCES = {
    'debate': 'الدليل من القرآن والسنة واضح، واللي يعترض يجيب دليله قدامنا',
    'joke': 'قالك مرة واحد راح يشتري عقل، قالوله خلص من زمان!',
    'rant': 'فوقوا بقى، البلد دي مش لعبة في إيديكم والناس زهقت!',
    'profanity': 'إنت وسخ يا معفن، فاكر نفسك مين يا ابن الكلب؟',
    'codeswitch': 'يا جماعة الـ live ده نار والـ chat بيضحك عليكم!',
}
# Backwards-compatible aliases (older cells referred to RANT/JOKE by name).
RANT = TEST_SENTENCES['rant']
JOKE = TEST_SENTENCES['joke']
DEBATE = TEST_SENTENCES['debate']
PROFANITY = TEST_SENTENCES['profanity']
CODESWITCH = TEST_SENTENCES['codeswitch']

for key, s in TEST_SENTENCES.items():
    n = len(s.split())
    print(f'{key:11s} ({n} words): {s}')
    assert n == 10, f'{key} is {n} words, not 10 — fix the sentence, do not change WORDS.'

WORDS = 10  # words per test sentence (the [laughter] tag in test #2 is not a word)
os.makedirs('/content/grid', exist_ok=True)
print('outputs go to /content/grid/grid_*.wav  (9 cells x 5 sentences = 45 WAVs)')


## §4 — The 9-cell grid (one cell per test; 5 sentences each)

Each cell writes one WAV to `/content/grid/` and prints what it did. **Run
them in order #1 → #9.** Model loaders are cached: VoiceTut is loaded once
for #1–#3, freed before base OmniVoice (#4), reloaded for #5–#7, then IndexTTS
loads in §4.8, then the emotion donor in §4.9. If a cell fails, read its error, fix that cell only, and re-run
just that cell — the other outputs stay on disk.

Filenames: `grid_01_voicetut_arz_rant_<sentence>.wav` … `grid_09_emotion_donor_<sentence>.wav` — 5 sentences per cell, 45 WAVs total.


In [ ]:
# --- §4 helpers: lazy model loaders + generate wrappers ------------------------
# Loaded once, cached in globals, freed explicitly when switching engines.
import gc, inspect
import soundfile as sf
import torch

# guidance_scale is HELD CONSTANT at 2.0 for every VoiceTut/OmniVoice cell
# (final-check §2.2): it is NOT a grid variable. VoiceTut's README example uses
# 2.5 and the GGUF default is 2.0 — the grid fixes 2.0. The wrappers below pass
# it only if the installed version exposes the parameter; if it does not, they
# print a NOTE so the run is recorded as guidance-uncontrolled, not silently
# assumed constant.
GUIDANCE_SCALE = 2.0

_VT = None
_OV = None

def free_models():
    global _VT, _OV
    _VT, _OV = None, None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('VRAM freed.')

def load_voicetut():
    global _VT
    if _VT is None:
        from voicetut_tts import VoiceTutTTS
        _VT = VoiceTutTTS.from_pretrained('mohammedaly22/VoiceTut-TTS')
        print('VoiceTut loaded. synthesize() params:',
              list(inspect.signature(_VT.synthesize).parameters))
    return _VT

def voicetut_synth(tts, text, ref_audio, ref_text, out_path, num_step=32, speed=1.0, language='arz'):
    """VoiceTut generate. `language` is passed only if the installed version
    exposes it; VoiceTut is Egyptian-only, so `arz` is its native behaviour and
    needs no parameter. The `arb` test (#3) is SKIPPED (not faked) if the
    version cannot select a language."""
    params = inspect.signature(tts.synthesize).parameters
    kw = dict(ref_audio=ref_audio, output=out_path, num_step=num_step, speed=speed)
    if 'guidance_scale' in params:
        kw['guidance_scale'] = GUIDANCE_SCALE
    else:
        print('⚠️ NOTE: this voicetut-tts version exposes no `guidance_scale` — guidance is uncontrolled in this run; record it in §6.')
    if ref_text is not None:
        kw['ref_text'] = ref_text  # omit (None) -> model auto-transcribes the ref clip
    if language is not None and language != 'arz':
        if 'language' in params:
            kw['language'] = language
        else:
            print(f'⚠️ SKIP: this voicetut-tts version has no `language` parameter —')
            print(f'   cannot run the {language} test as specified. Leaving {out_path} ungenerated;')
            print('   score it as "not run" in §6. (VoiceTut is Egyptian-only, so `arz` is its default.)')
            return None
    tts.synthesize(text, **kw)
    print('wrote', out_path, f'(num_step={num_step}, speed={speed}, language={language})')
    return out_path

def load_omnivoice():
    global _OV
    if _OV is None:
        from omnivoice import OmniVoice
        _OV = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
        print('Base OmniVoice loaded. generate() params:',
              list(inspect.signature(_OV.generate).parameters))
    return _OV

def omni_generate(model, text, ref_audio, ref_text, out_path, language_id='arz', num_step=32, speed=1.0):
    params = inspect.signature(model.generate).parameters
    kw = dict(text=text, ref_audio=ref_audio, num_step=num_step)
    if 'guidance_scale' in params:
        kw['guidance_scale'] = GUIDANCE_SCALE
    else:
        print('⚠️ NOTE: this OmniVoice version exposes no `guidance_scale` — guidance is uncontrolled in this run; record it in §6.')
    if ref_text is not None:
        kw['ref_text'] = ref_text
    if 'language_id' in params:
        kw['language_id'] = language_id
    if 'speed' in params:
        kw['speed'] = speed
    audio = model.generate(**kw)
    sf.write(out_path, audio[0], 24000)
    print('wrote', out_path, f'(language_id={language_id}, num_step={num_step}, speed={speed})')
    return out_path


In [ ]:
# --- Test #1: VoiceTut | arz | fast-rant ref | manual ref_text | speed 1.0 | num_step 32 | guidance_scale 2.0 ---
# Answers: does Egyptian fine-tuning fix dialect + pace?
# Same 5 fixed sentences as every other cell (one WAV per sentence).
vt = load_voicetut()
for key, text in TEST_SENTENCES.items():
    voicetut_synth(vt, text, REF_RANT, REF_RANT_TEXT,
                   f'/content/grid/grid_01_voicetut_arz_rant_{key}.wav',
                   num_step=32, speed=1.0, language='arz')


In [ ]:
# --- Test #2: VoiceTut | arz | LAUGHING ref | manual ref_text | speed 1.0 | num_step 32 | guidance_scale 2.0 ---
# Answers: does ref style transfer to output? (+ [laughter] tag test)
# The [laughter] tag prefixes the JOKE line only; the other 4 sentences run
# plain, so #2 stays comparable with the other cells. OPEN TEST (see §3):
# plain VoiceTut may READ the tag aloud instead of laughing — log the
# behaviour per cell in §6 (read aloud / ignored / laughed), do not assume it.
vt = load_voicetut()
for key, text in TEST_SENTENCES.items():
    t = '[laughter] ' + text if key == 'joke' else text
    voicetut_synth(vt, t, REF_LAUGH, REF_LAUGH_TEXT,
                   f'/content/grid/grid_02_voicetut_arz_laugh_{key}.wav',
                   num_step=32, speed=1.0, language='arz')


In [ ]:
# --- Test #3: VoiceTut | arb | same ref as #1 | manual ref_text | speed 1.0 | num_step 32 | guidance_scale 2.0 ---
# Answers: is arz vs arb the dialect difference?
# If the installed voicetut-tts has no `language` parameter, this cell SKIPS
# itself (see helper) — do not treat a missing file as a bad score.
# Same 5 fixed sentences as every other cell.
vt = load_voicetut()
for key, text in TEST_SENTENCES.items():
    voicetut_synth(vt, text, REF_RANT, REF_RANT_TEXT,
                   f'/content/grid/grid_03_voicetut_arb_rant_{key}.wav',
                   num_step=32, speed=1.0, language='arb')


In [ ]:
# --- Test #4: BASE OmniVoice | arz | same ref as #1 | manual ref_text | speed 1.0 | num_step 32 | guidance_scale 2.0 ---
# Answers: is VoiceTut actually better than base?
# VoiceTut is freed first (VRAM), then the base k2-fsa/OmniVoice model loads.
# Same 5 fixed sentences as every other cell.
free_models()
ov = load_omnivoice()
for key, text in TEST_SENTENCES.items():
    omni_generate(ov, text, REF_RANT, REF_RANT_TEXT,
                  f'/content/grid/grid_04_omnivoice_arz_rant_{key}.wav',
                  language_id='arz', num_step=32, speed=1.0)


In [ ]:
# --- Test #5: VoiceTut | arz | same ref as #1 | AUTO ref_text (omitted) | speed 1.0 | num_step 32 | guidance_scale 2.0 ---
# Answers: how much does Whisper mis-transcription cost?
# ref_text is deliberately NOT passed: the model auto-transcribes the rant clip
# with Whisper. Compare with #1 — if #5 is clearly worse, always use manual ref_text.
# Same 5 fixed sentences as every other cell.
free_models()
vt = load_voicetut()
for key, text in TEST_SENTENCES.items():
    voicetut_synth(vt, text, REF_RANT, None,
                   f'/content/grid/grid_05_voicetut_arz_rant_auto_{key}.wav',
                   num_step=32, speed=1.0, language='arz')


In [ ]:
# --- Test #6: VoiceTut | arz | same ref as #1 | manual ref_text | speed 1.25 | num_step 32 | guidance_scale 2.0 ---
# Answers: does speed help, or only trim?
# Prior finding: the "slow" feeling was ~50% dead air, not slow words — so a
# faster speed may help the feel, or may just rush the words. Judge by ear + §5.
# Same 5 fixed sentences as every other cell.
vt = load_voicetut()
for key, text in TEST_SENTENCES.items():
    voicetut_synth(vt, text, REF_RANT, REF_RANT_TEXT,
                   f'/content/grid/grid_06_voicetut_arz_rant_speed125_{key}.wav',
                   num_step=32, speed=1.25, language='arz')


In [ ]:
# --- Test #7: VoiceTut | arz | same ref as #1 | manual ref_text | speed 1.0 | num_step 16 | guidance_scale 2.0 ---
# Answers: quality cost of fast sampling?
# num_step trades quality for speed, NOT pace (16 = faster/lower quality,
# 32 = default/balanced). If #7 ≈ #1 by ear, use 16 for faster batch generation.
# Same 5 fixed sentences as every other cell.
vt = load_voicetut()
for key, text in TEST_SENTENCES.items():
    voicetut_synth(vt, text, REF_RANT, REF_RANT_TEXT,
                   f'/content/grid/grid_07_voicetut_arz_rant_step16_{key}.wav',
                   num_step=16, speed=1.0, language='arz')
free_models()


## §4.8 — Test #8: IndexTTS 2.5 (third engine)

IndexTTS 2.5 (released 2026-08-10, repo `index-tts/index-tts`) is the best
prosody surface in the field: separate **emotion reference audio**
(`emo_audio_prompt`) + intensity `emo_alpha` 0.0–1.0, and precise speed via
`duration_factor` 0.5–2.0 (<1.0 = faster). Arabic (AR) is supported in 2.5 —
but that is generic Arabic, almost certainly MSA, **not `arz`** — so expect
Fusha-leaning pronunciation even if the emotion/pace is good. ~6 GB VRAM BF16,
fits a T4.

- **Install (repo-documented `uv` path):** `pip install -U uv`, clone the repo,
  `uv sync --all-extras` (~10 min), weights `IndexTeam/IndexTTS-2.5` from
  Hugging Face. Inference runs with the repo's own venv python
  (`.venv/bin/python`), **not** Colab's python.
- **Test #8:** timbre ref = rant clip `bahgat_0621.wav`, emotion ref = the
  **same shouting rant clip** (the emotion IS the shout), `emo_alpha=1.0`,
  `duration_factor=0.8` (faster), `lang="AR"` (retries `"ar"` if rejected).
  Alternate emotion ref if you want a distinct one: the loud call clip
  `bahgat_0265.wav` (RMS −11.8 dB) — swap `EMO` in the script.

### ⚠️ IndexTTS pitfalls (final-check §2.2 — read before scoring #8)

- **`duration_factor` is INVERTED vs VoiceTut `speed`:** `>1.0` **slows**,
  `<1.0` **speeds** (`0.8` = faster). Do not read it as a speed multiplier —
  `duration_factor=1.25` would make Bahgat slower, not faster.
- **`use_emo_text=True` requires `use_qwen_emo=True`** — without it,
  `infer` raises a `RuntimeError`. If you switch #8 from an emotion *audio*
  prompt to an emotion *text* prompt, set both flags together.
- **`emo_alpha` ~0.6 or lower is recommended for natural speech** when using
  `use_emo_text`. Test #8 below uses an emotion *audio* prompt
  (`emo_audio_prompt`) with `emo_alpha=1.0` — that is the audit's deliberate
  choice for a shouting emotion ref, not the `use_emo_text` case. If you
  switch #8 to `use_emo_text`, drop `emo_alpha` to ≤0.6 at the same time.
- **Arabic quality is unproven:** IndexTTS 2.5's model card cites no benchmark
  numbers for Arabic — treat its Arabic as untested until this grid hears it.


In [ ]:
# --- IndexTTS 2.5 install (uv-managed; ~10 min, run once) ---------------------
# If this cell fails, tests #1-#7 are unaffected — fix it or skip #8.
!pip install -q -U uv 2>&1 | tail -n 2
!git clone https://github.com/index-tts/index-tts.git /content/index-tts 2>&1 | tail -n 2
%cd /content/index-tts
!uv sync --all-extras 2>&1 | tail -n 5
!ls /content/index-tts/.venv/bin/python && /content/index-tts/.venv/bin/python --version


In [ ]:
# --- IndexTTS 2.5 weights: IndexTeam/IndexTTS-2.5 --------------------------------
# snapshot_download first; falls back to the `hf` CLI if the import is missing.
%cd /content/index-tts
import os
try:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='IndexTeam/IndexTTS-2.5', local_dir='checkpoints')
except Exception as e:
    print('snapshot_download failed:', e)
    print('trying hf CLI ...')
    import subprocess
    r = subprocess.run(['hf', 'download', 'IndexTeam/IndexTTS-2.5', '--local-dir=checkpoints'],
                       capture_output=True, text=True)
    print(r.stdout[-1000:]); print(r.stderr[-1000:])
print('checkpoints:', os.listdir('checkpoints') if os.path.exists('checkpoints') else 'MISSING')
assert os.path.exists('checkpoints/config.yaml'), 'config.yaml missing — weights did not download.'


In [ ]:
# --- Test #8: IndexTTS 2.5 | AR | rant (timbre) + shouting rant (emotion) | duration_factor 0.8 | emo_alpha 1.0 ---
# Same 5 fixed sentences as every other cell (one WAV per sentence).
# Answers: best emotion/duration control?
# Runs in the repo venv (uv environment). Non-fatal: if it fails, the cell
# prints the error and moves on — tests #1-#7 still stand.
import subprocess

SCRIPT = '''
import sys
from indextts.infer_v2_5 import IndexTTS2

REF = "/content/refs/bahgat_0621.wav"  # timbre ref: fast shouting rant
EMO = "/content/refs/bahgat_0621.wav"  # emotion ref: same shouting rant (swap to bahgat_0265.wav for the loud call clip)
TEXT = sys.argv[1]
OUT = sys.argv[2]  # per-sentence output path, passed by the loop below

tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)
try:
    tts.infer(spk_audio_prompt=REF, text=TEXT, lang="AR", output_path=OUT,
              emo_audio_prompt=EMO, emo_alpha=1.0, duration_factor=0.8, verbose=True)
except Exception as e:
    print("lang=AR failed:", e)
    print("retrying with lang=ar ...")
    tts.infer(spk_audio_prompt=REF, text=TEXT, lang="ar", output_path=OUT,
              emo_audio_prompt=EMO, emo_alpha=1.0, duration_factor=0.8, verbose=True)
print("wrote", OUT)
'''
open('/content/index-tts/run_grid08.py', 'w', encoding='utf-8').write(SCRIPT)

for key, text in TEST_SENTENCES.items():
    out = f'/content/grid/grid_08_indextts25_ar_rant_{key}.wav'
    r = subprocess.run(['/content/index-tts/.venv/bin/python', 'run_grid08.py', text, out],
                       cwd='/content/index-tts', capture_output=True, text=True)
    print(f'--- {key}: returncode {r.returncode}')
    print(r.stdout[-1500:])
    print(r.stderr[-1500:])
    if r.returncode != 0:
        print(f'⚠️ IndexTTS test #8 ({key}) FAILED (returncode {r.returncode}) — see output above.')
        print('   The other sentences/cells still stand. Note it as "not run" in §6 and move on.')


## §4.9 — Test #9: `voicetut-egyptian-emotion` — DONOR ONLY (new, final-check §1/§2.2)

**What it is:** `masterofaudio2077/voicetut-egyptian-emotion` — a full
VoiceTut fine-tune on 19,480 Egyptian clips (46.4 h), **one voice, 20
emotions**, with tags such as `[shouting]` and `[furious, barely holding it
together]`. Held-out results on its card: CER 0.025 vs VoiceTut 0.192
(Whisper large-v3), speaker similarity 0.960 vs 0.539, loudness range 4.7 dB
vs 2.9 dB, pitch range 109 Hz vs 23 Hz. Usage below is copied from the live
model card (verified 2026-11-26): `snapshot_download(...)`, then
`from omni_emotion import generate, seconds` and
`generate(model, text, seconds(text, emotion, rates))`.

**Why it matters — loudness-preserving tokenisation:** it was trained with
audio "tokenised at each clip's own level (OmniVoice's default normalises it,
which erases loudness)" and its `generate()` "keeps the output level, so
whispers stay quiet and shouts stay loud". **If #9 wins on shouting, that
mechanism is the reason to record in the scorecard** — it is the one thing
plain VoiceTut/OmniVoice cannot do by construction. The card also warns that
OmniVoice's own `model.generate` "counts the English tag as speech and often
reads it aloud" — which is why #9 uses the card's own `generate()`, never
`model.generate()`.

**⛔ DONOR ONLY — it can NEVER be the Bahgat voice itself.** Verbatim from
the card's Limitations: "One voice only. Cloning another voice from a
reference clip is not supported. The training audio is synthetic (Fish Audio
Drama 3)… Not for impersonating a real person." There is **no `ref_audio`**
in this cell on purpose. Its outputs are **raw donor WAVs** (same 5 fixed
test sentences as every other cell) to be **RVC re-timbred** to Bahgat
afterwards — judge #9 on **prosody/pace/loudness only**, never on timbre.

**Tags used (one per sentence, from the card's Tags list):** debate →
`[furious, barely holding it together]` (`angry`); joke → `[playful and
teasing]` (`playful`); rant + profanity → `[shouting]` (`shouting`);
code-switch → `[with bright eager excitement]` (`excited`).


In [ ]:
# --- Test #9: voicetut-egyptian-emotion | DONOR ONLY | arz | emotion tags ---
# DONOR ONLY: single synthetic voice, cloning unsupported (HF card, Limitations).
# It can NEVER be the Bahgat voice itself — judge prosody/pace/loudness only,
# then RVC re-timbre the winner. Same 5 fixed sentences as every other cell.
# Usage verified against https://huggingface.co/masterofaudio2077/voicetut-egyptian-emotion
# (card: `from omni_emotion import generate, seconds`; generate() keeps output
# level — whispers stay quiet, shouts stay loud — because audio was tokenised
# at each clip's own level, unlike OmniVoice's default normalisation).
import json, os
import soundfile as sf
import torch
from huggingface_hub import snapshot_download
from omnivoice import OmniVoice

free_models()
EMO_PATH = snapshot_download('masterofaudio2077/voicetut-egyptian-emotion')
assert os.path.exists(f'{EMO_PATH}/omni_emotion.py'), \
    f'omni_emotion.py not in the snapshot at {EMO_PATH} — the card API moved; check the live model card before re-running.'
import sys; sys.path.insert(0, EMO_PATH)
from omni_emotion import generate, seconds

emo_model = OmniVoice.from_pretrained(EMO_PATH, device_map='cuda:0', dtype=torch.float16)
rates = json.load(open(f'{EMO_PATH}/rates.json'))

# (emotion name for seconds(), tag prepended to the text) — tags from the card.
EMO_TAGS = {
    'debate': ('angry', '[furious, barely holding it together]'),
    'joke': ('playful', '[playful and teasing]'),
    'rant': ('shouting', '[shouting]'),
    'profanity': ('shouting', '[shouting]'),
    'codeswitch': ('excited', '[with bright eager excitement]'),
}
for key, text in TEST_SENTENCES.items():
    emo, tag = EMO_TAGS[key]
    tagged = f'{tag} {text}'
    audio = generate(emo_model, tagged, seconds(tagged, emo, rates))
    out = f'/content/grid/grid_09_emotion_donor_{key}.wav'
    sf.write(out, audio, 24000)
    print(f'wrote {out} (emotion={emo}, tag={tag})')
del emo_model
torch.cuda.empty_cache()
print('NOTE: these are DONOR wavs — raw prosody to RVC-re-timbre, NOT a Bahgat voice.')
print('If #9 wins on shouting, record the mechanism in §6: loudness-preserving tokenisation.')


## §5 — Measurement (numbers, no Whisper needed)

Same measurement as the earlier OmniVoice check, so the numbers are comparable:

- **Words/s on speech only** = 10 words ÷ speech seconds (total duration minus
  silence). The earlier baseline: real Bahgat **1.98 w/s** on speech only;
  the first OmniVoice sample was **2.82 w/s** but **~50% dead air** — fast
  words, slow *feel*. So watch the **% silence** column, not just words/s.
- **% silence** from `ffmpeg silencedetect` (noise −30 dB, min 0.1 s). If a
  sample is >40% silence, the fix is trimming pauses, **not** raising speed.

No Whisper install is needed here: the word count (10) is known because the
test sentences are fixed (§3) — every grid file is one 10-word sentence, so
words/s is comparable across all 45 files (9 cells × 5 sentences).


In [ ]:
# --- Measure every grid output: duration, % silence, words/s on speech only ----
import glob, json, os, re, subprocess
import soundfile as sf

def silence_seconds(path):
    r = subprocess.run(['ffmpeg', '-i', path, '-af', 'silencedetect=noise=-30dB:d=0.1',
                        '-f', 'null', '-'], capture_output=True, text=True)
    return sum(float(x) for x in re.findall(r'silence_duration: ([0-9.]+)', r.stderr))

rows = []
for p in sorted(glob.glob('/content/grid/grid_*.wav')):
    info = sf.info(p)
    sil = silence_seconds(p)
    speech = max(info.duration - sil, 0.01)
    sentence = os.path.basename(p).replace('.wav', '').rsplit('_', 1)[-1]
    rows.append({'file': os.path.basename(p), 'sentence': sentence,
                 'duration_s': round(info.duration, 2),
                 'silence_pct': round(100 * sil / info.duration, 1),
                 'words_per_s_speech_only': round(WORDS / speech, 2)})

print(f"{'file':42s} {'sentence':>11s} {'dur':>6s} {'sil%':>5s} {'w/s':>5s}")
for r in rows:
    print(f"{r['file']:42s} {r['sentence']:>11s} {r['duration_s']:6.2f} {r['silence_pct']:5.1f} {r['words_per_s_speech_only']:5.2f}")
print()
print('Reference: real Bahgat 1.98 w/s on speech only; first OmniVoice sample 2.82 w/s with ~50% silence.')
print('Read it as: high w/s + high silence% = fast words, slow feel (trim pauses, do not raise speed).')
json.dump(rows, open('/content/grid/measurements.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('saved /content/grid/measurements.json')


## §6 — Scorecard (fill in BY EAR — this is the verdict)

Listen to the reference clips first, then #1–#9 in order (the playback cell is
below — all 5 sentences per cell). Score each **1–5 (5 = best)**. Judge the
**voice**, not the sentence — everyone says the same words. Score **dialect,
intelligibility, speed, laughter and loudness separately** (final-check §2.2):
a line can be intelligible but Fusha-leaning, or loud but unintelligible.

| # | Test | Timbre 1–5 (sounds like Bahgat?) | Dialect 1–5 (Egyptian, not Fusha?) | Pace 1–5 (fast/loud feel, no dead air?) | Tag behaviour (read aloud / ignored / laughed) | Notes (artefacts, mispronounced words, laugh worked?) |
|---|---|---|---|---|---|---|
| 1 | VoiceTut arz rant |  |  |  | — |  |
| 2 | VoiceTut arz laugh (+`[laughter]` on joke) |  |  |  | **read aloud / ignored / laughed?** |  |
| 3 | VoiceTut arb rant |  |  |  | — |  |
| 4 | Base OmniVoice arz rant |  |  |  | — |  |
| 5 | VoiceTut arz rant, auto ref_text |  |  |  | — |  |
| 6 | VoiceTut arz rant, speed 1.25 |  |  |  | — |  |
| 7 | VoiceTut arz rant, step 16 |  |  |  | — |  |
| 8 | IndexTTS 2.5 AR rant |  |  |  | — |  |
| 9 | Emotion donor (tag-only, **donor — score prosody/loudness, NOT timbre**) | — |  |  | **tag read aloud / ignored / felt?** |  |

**`[laughter]` / tag behaviour — open test, not an assumed feature:** plain
VoiceTut/OmniVoice may **read the tag aloud** (the emotion-model card warns
OmniVoice's `generate` "counts the English tag as speech and often reads it
aloud"). Log what actually happened per cell in the column above — **read
aloud / ignored / laughed** — instead of assuming tags work.

**Per-sentence notes (one line each — which cell handled this sentence best,
and what broke?):**
- debate: …
- joke: …
- rant: …
- profanity: …
- code-switch: …

**Verdict — apply the header's decision rule and write one line:**
- If #1/#2 are close on timbre **and** pace (≥4/5 both) → **skip RVC training**
  (or use VoiceTut as the RVC donor). Otherwise → run the RVC training
  notebook.
- If #9 wins on **shouting prosody/loudness** → use it as the **RVC donor**
  for rant/shouting lines, and record the mechanism: **loudness-preserving
  tokenisation** (audio tokenised at each clip's own level, so shouts stay
  loud). Never score #9 on timbre — it is a single synthetic voice, not Bahgat.
- What specifically to listen for: Bahgat is loud, fast, raspy, aggressive —
  a flat or polite delivery scores low on timbre even if it is clear. A
  Fusha-leaning pronunciation scores low on dialect even if it is intelligible.
  Dead air between words scores low on pace even if the words themselves are fast.


In [ ]:
# --- Listen: references first, then #1-#9 in order (all 5 sentences per cell) ---
from IPython.display import Audio, display
import os

def show(label, path):
    import soundfile as sf
    if not os.path.exists(path):
        print(f'{label}: MISSING (not run / skipped) — {os.path.basename(path)}')
        return
    info = sf.info(path)
    print(f'{label}: {os.path.basename(path)} — {info.duration:.2f} s, {info.samplerate} Hz')
    display(Audio(path))

show('0a — REFERENCE rant (real Bahgat)', REF_RANT)
show('0b — REFERENCE laughing (real Bahgat)', REF_LAUGH)
show('0c — REFERENCE call (real Bahgat)', REF_CALL)
CELL_FILES = {
    '01': ('VoiceTut arz rant', 'voicetut_arz_rant'),
    '02': ('VoiceTut arz laugh +[laughter] on joke', 'voicetut_arz_laugh'),
    '03': ('VoiceTut arb rant', 'voicetut_arb_rant'),
    '04': ('Base OmniVoice arz rant', 'omnivoice_arz_rant'),
    '05': ('VoiceTut arz rant auto-ref', 'voicetut_arz_rant_auto'),
    '06': ('VoiceTut arz rant speed 1.25', 'voicetut_arz_rant_speed125'),
    '07': ('VoiceTut arz rant step 16', 'voicetut_arz_rant_step16'),
    '08': ('IndexTTS 2.5 AR rant', 'indextts25_ar_rant'),
    '09': ('Emotion donor (DONOR ONLY — prosody, not timbre)', 'emotion_donor'),
}
for num, (label, stem) in CELL_FILES.items():
    print(f'\n=== #{num} — {label} ===')
    for key in ['debate', 'joke', 'rant', 'profanity', 'codeswitch']:
        show(f'{num} {key}', f'/content/grid/grid_{num}_{stem}_{key}.wav')


## §7 — Export everything to Drive

Saves all **45 outputs (9 cells × 5 sentences)** + the 3 reference clips +
`measurements.json` + a blank `scorecard.md` + `grid_notes.json` (settings per
test) to
`My Drive/bahgat_voice_models/voicetut_grid_2026-11-26/`. The VM disk is wiped
when the session ends — **export before closing the tab**.

These are the **raw donor WAVs** — saved here **before any RVC pass**. RVC
re-timbre happens later, in the training notebook's workflow, never here.


In [ ]:
# --- Export grid outputs + scorecard to Drive ------------------------------------
import datetime, glob, json, os, shutil, subprocess

OUT_DIR = '/content/drive/MyDrive/bahgat_voice_models/voicetut_grid_2026-11-26'
os.makedirs(OUT_DIR, exist_ok=True)

for f in sorted(glob.glob('/content/grid/grid_*.wav')):
    shutil.copy(f, os.path.join(OUT_DIR, os.path.basename(f)))
for f in ['/content/refs/bahgat_0621.wav', '/content/refs/bahgat_0470.wav', '/content/refs/bahgat_0265.wav',
          '/content/refs/ref_transcripts.json', '/content/grid/measurements.json']:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(OUT_DIR, os.path.basename(f)))

SCORECARD = '''# VoiceTut grid scorecard — 2026-11-26 (fill by ear, 1-5)

| # | Test | Timbre | Dialect | Pace | Tag behaviour (read aloud / ignored / laughed) | Notes |
|---|---|---|---|---|---|---|
| 1 | VoiceTut arz rant |  |  |  | — |  |
| 2 | VoiceTut arz laugh +[laughter] on joke |  |  |  |  |  |
| 3 | VoiceTut arb rant |  |  |  | — |  |
| 4 | Base OmniVoice arz rant |  |  |  | — |  |
| 5 | VoiceTut arz rant auto-ref |  |  |  | — |  |
| 6 | VoiceTut arz rant speed 1.25 |  |  |  | — |  |
| 7 | VoiceTut arz rant step 16 |  |  |  | — |  |
| 8 | IndexTTS 2.5 AR rant |  |  |  | — |  |
| 9 | Emotion donor (DONOR ONLY — prosody/loudness, NOT timbre) | — |  |  |  |  |

Per-sentence best cell (one line each): debate: ... | joke: ... | rant: ... | profanity: ... | code-switch: ...

Verdict (one line): ...
'''
open(os.path.join(OUT_DIR, 'scorecard.md'), 'w', encoding='utf-8').write(SCORECARD)

notes = {
    'date': '2026-11-26',
    'test_sentences': {k: v for k, v in TEST_SENTENCES.items()} | {'words_each': WORDS},
    'guidance_scale': GUIDANCE_SCALE,  # held constant for all VoiceTut/OmniVoice cells
    'references': {
        'rant': {'file': 'bahgat_0621.wav', 'duration_s': 9.30, 'f0_hz': 193.2, 'ref_text': REF_RANT_TEXT},
        'laughing': {'file': 'bahgat_0470.wav', 'duration_s': 7.68, 'f0_hz': 172.9, 'ref_text': REF_LAUGH_TEXT},
        'call': {'file': 'bahgat_0265.wav', 'duration_s': 8.30, 'f0_hz': 154.3, 'ref_text': REF_CALL_TEXT}},
    'grid': {
        '01': {'model': 'VoiceTut-TTS (mohammedaly22/VoiceTut-TTS)', 'language': 'arz', 'ref': 'rant', 'ref_text': 'manual', 'speed': 1.0, 'num_step': 32},
        '02': {'model': 'VoiceTut-TTS', 'language': 'arz', 'ref': 'laughing', 'ref_text': 'manual', 'speed': 1.0, 'num_step': 32, 'tag': '[laughter] + joke'},
        '03': {'model': 'VoiceTut-TTS', 'language': 'arb', 'ref': 'rant', 'ref_text': 'manual', 'speed': 1.0, 'num_step': 32},
        '04': {'model': 'Base OmniVoice (k2-fsa/OmniVoice)', 'language': 'arz', 'ref': 'rant', 'ref_text': 'manual', 'speed': 1.0, 'num_step': 32},
        '05': {'model': 'VoiceTut-TTS', 'language': 'arz', 'ref': 'rant', 'ref_text': 'auto (omitted)', 'speed': 1.0, 'num_step': 32},
        '06': {'model': 'VoiceTut-TTS', 'language': 'arz', 'ref': 'rant', 'ref_text': 'manual', 'speed': 1.25, 'num_step': 32},
        '07': {'model': 'VoiceTut-TTS', 'language': 'arz', 'ref': 'rant', 'ref_text': 'manual', 'speed': 1.0, 'num_step': 16},
        '08': {'model': 'IndexTTS 2.5 (IndexTeam/IndexTTS-2.5)', 'language': 'AR', 'ref': 'rant timbre + shouting rant emotion', 'duration_factor': 0.8, 'emo_alpha': 1.0},
        '09': {'model': 'voicetut-egyptian-emotion (masterofaudio2077/voicetut-egyptian-emotion)', 'language': 'arz', 'ref': 'none (single voice, tag-only)', 'ref_text': 'n/a', 'role': 'DONOR ONLY — RVC re-timbre required, never the Bahgat voice itself', 'tags': {'debate': '[furious, barely holding it together]', 'joke': '[playful and teasing]', 'rant': '[shouting]', 'profanity': '[shouting]', 'codeswitch': '[with bright eager excitement]'}}},
    'scores': {f'{i:02d}': {'timbre_1_5': None, 'dialect_1_5': None, 'pace_1_5': None, 'tag_behaviour': None, 'notes': ''} for i in range(1, 10)},
    'verdict': None,
}
json.dump(notes, open(os.path.join(OUT_DIR, 'grid_notes.json'), 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)

print('Exported to:', OUT_DIR)
print(subprocess.run(['ls', '-la', OUT_DIR], capture_output=True, text=True).stdout)
print()
print('NEXT: fill scorecard.md by ear. If VoiceTut (#1/#2) is close on timbre AND pace,')
print('skip RVC training (or use VoiceTut as the RVC donor). If the emotion donor (#9)')
print('wins on shouting prosody/loudness, use it as the RVC donor for rant lines')
print('(RVC re-timbre — never its own voice). Otherwise open')
print('bahgat_rvc_gpt-sovits_training.ipynb — the Drive upload is already done.')
